# Grid Parameter Comparison Dashboard

Compare synthetic grids (from `grid_parameters` database table) with real grids (from `real_grid_metrics.csv`).

Interactive dashboard to explore grid metrics.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
from pathlib import Path
from pylovo.database.database_client import DatabaseClient

# Set Plotly Template
import plotly.io as pio
pio.templates.default = "plotly_white"

## 0. Configure Data Selection

In [ ]:
metrics = [
    "feeder_lines", 
    "house_connections", 
    "cable_length", 
    "avg_trafo_distance", 
    "max_trafo_distance"
 ]

labels = {
    "feeder_lines": "Number of Feeder Lines (count)",
    "house_connections": "Number of House Connections (count)",
    "cable_length": "Total Cable Length (km)",
    "avg_trafo_distance": "Avg. Distance to Trafo (km)",
    "max_trafo_distance": "Max. Distance to Trafo (km)"
}

## 1. Load Data

In [3]:
# 1. Load Synthetic and Real Data from CSV if available, else fall back to DB for synthetic data
def resolve_metrics_path(filename: str) -> Path | None:
    candidates = [
        Path("validation") / "metrics" / filename,
        Path("metrics") / filename,
        Path("..") / "validation" / "metrics" / filename,
        Path("..") / "metrics" / filename,
    ]
    
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


synth_csv_path = resolve_metrics_path("synthetic_grid_metrics.csv")
if synth_csv_path is not None:
    df_synth = pd.read_csv(synth_csv_path)
    print(f"Loaded {len(df_synth)} synthetic grids from CSV: {synth_csv_path}")
else:
    try:
        with DatabaseClient() as dbc:
            query = "SELECT * FROM grid_parameters"
            df_synth = pd.read_sql(query, dbc.sqla_engine)
            print(f"Loaded {len(df_synth)} synthetic grids from DB.")
    except Exception as e:
        print(f"Error loading synthetic data: {e}")
        df_synth = pd.DataFrame()

if not df_synth.empty:
    df_synth['Type'] = 'Synthetic'


real_csv_path = resolve_metrics_path("real_grid_metrics.csv")
if real_csv_path is not None:
    df_real = pd.read_csv(real_csv_path)
    df_real['Type'] = 'Real'
    print(f"Loaded {len(df_real)} real grids from CSV: {real_csv_path}")
else:
    print("Real grid metrics CSV not found.")
    df_real = pd.DataFrame()


df_all = pd.concat([df_synth, df_real], ignore_index=True)
# df_all.head()

Loaded 119 synthetic grids from CSV: metrics/synthetic_grid_metrics.csv
Loaded 86 real grids from CSV: metrics/real_grid_metrics.csv


## 2. Structural Comparison

In [4]:
def plot_comparison(metric):
    if metric not in df_all.columns:
        print(f"Metric '{metric}' not found in data.")
        return
        
    fig = make_subplots(rows=1, cols=2, 
                        subplot_titles=("Boxplot", "Distribution (PDF)"))

    colors = {'Synthetic': 'blue', 'Real': 'red'}
    
    for grid_type in ['Synthetic', 'Real']:
        subset = df_all[df_all['Type'] == grid_type]
        data = subset[metric].dropna()
        color = colors[grid_type]
        
        if len(data) == 0:
            continue

        # 1. Boxplot
        fig.add_trace(
            go.Box(y=data, name=grid_type, marker_color=color, legendgroup=grid_type),
            row=1, col=1
        )
        
        # 2. Histogram / PDF
        fig.add_trace(
            go.Histogram(x=data, name=grid_type, marker_color=color, opacity=0.5, 
                         histnorm='probability density', legendgroup=grid_type, showlegend=False),
            row=1, col=2
        )
        
        # KDE Line (manual implementation) - Commented out as per request
        # if len(data) > 1 and data.std() > 0:
        #     try:
        #         kde = stats.gaussian_kde(data)
        #         x_range = np.linspace(data.min(), data.max(), 100)
        #         y_kde = kde(x_range)
        #         
        #         fig.add_trace(
        #             go.Scatter(x=x_range, y=y_kde, mode='lines', 
        #                        name=f'{grid_type} KDE', 
        #                        line=dict(color=color, width=2),
        #                        legendgroup=grid_type, showlegend=False),
        #             row=1, col=2
        #         )
        #     except Exception as e:
        #         print(f"Could not plot KDE for {grid_type}: {e}")

    fig.update_layout(
        title_text=f"Comparison: {labels.get(metric, metric)}",
        barmode='overlay',
        height=500,
        legend_title_text='Grid Type'
    )
    fig.update_xaxes(title_text=labels.get(metric, metric), row=1, col=2)
    fig.update_yaxes(title_text=labels.get(metric, metric), row=1, col=1)
    fig.update_yaxes(title_text="Probability Density", row=1, col=2)
    
    fig.show()

# Interactive Widget
interact(plot_comparison, metric=widgets.Dropdown(options=metrics, description="Parameter:"));

interactive(children=(Dropdown(description='Parameter:', options=('feeder_lines', 'house_connections', 'cable_…

## 3. Violin Plots

In [ ]:
metrics = [
    "feeder_lines", 
    "house_connections", 
    "cable_length", 
    "avg_trafo_distance", 
    "max_trafo_distance"
 ]

labels = {
    "feeder_lines": "Number of Feeder Lines (count)",
    "house_connections": "Number of House Connections (count)",
    "cable_length": "Total Cable Length (km)",
    "avg_trafo_distance": "Avg. Distance to Trafo (km)",
    "max_trafo_distance": "Max. Distance to Trafo (km)"
}

def plot_comparison(metric):
    if metric not in df_all.columns:
        print(f"Metric '{metric}' not found in data.")
        return
        
    # Violin Plot - good statistical overview
    fig = go.Figure()

    colors = {'Synthetic': 'blue', 'Real': 'red'}
    
    for grid_type in ['Synthetic', 'Real']:
        subset = df_all[df_all['Type'] == grid_type]
        data = subset[metric].dropna()
        
        if len(data) == 0:
            continue
            
        fig.add_trace(go.Violin(
            y=data,
            x0='Comparison', # <--- Force them to the same X position
            name=grid_type,
            scalegroup='group1', # <--- Ensure widths are scaled together
            box_visible=True,
            meanline_visible=True,
            line_color=colors[grid_type],
            points='all',
            jitter=0.05,
            pointpos=-1.8 if grid_type == 'Synthetic' else 1.8,
            side='negative' if grid_type == 'Synthetic' else 'positive',
        ))

    fig.update_layout(
        # ... your other layout settings ...
        violinmode='overlay',
        violingap=0,
        violingroupgap=0 # <--- Added to remove any remaining spacing
    )
    
    fig.show()

# Interactive Widget
interact(plot_comparison, metric=widgets.Dropdown(options=metrics, description="Parameter:"));

interactive(children=(Dropdown(description='Parameter:', options=('feeder_lines', 'house_connections', 'cable_…